# Robust HBI Trade Map
Build a robust HBI map from three scenarios by taking the element-wise minimum of HBI trade links and HBI production links, while keeping demand unchanged.

In [ ]:
import os
import re
import sys
from pathlib import Path

import pandas as pd
import pypsa

if "snakemake" not in globals():
    from _helpers_notebooks import mock_snakemake

    snakemake = mock_snakemake(
        "plot_robust_map",
    )

scripts_dir = Path.cwd() / "workflow" / "scripts"
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

import model_trade as mt

mt.snakemake = snakemake
trade_chain = snakemake.config["trade_chains"]
mt.final = trade_chain["final_product"]
tradeable = trade_chain["tradeable_commodities"]
mt.interone = next(c for c in tradeable if c != "iron_ore")
stages = trade_chain["stages"]
mt.intertwo = stages[max(stages.keys())]["process_label"]

In [ ]:
scenario_input_paths = {
    "chokepoints": snakemake.input.chokepoints,
    "blocs": snakemake.input.blocs,
    "constrain_supply": snakemake.input.constrain_supply,
}


def resolve_network_input(path):
    # Prefer direct netCDF input; keep a backward-compatible fallback for old PDF-style inputs.
    if path.endswith(".nc") and os.path.exists(path):
        return path

    if path.endswith(".pdf"):
        filename = os.path.basename(path)
        match = re.match(r"^map_hbi(?:_(.+))?\.pdf$", filename)
        if match:
            suffix = match.group(1)
            candidate = os.path.join(
                os.path.dirname(path),
                "network.nc" if suffix is None else f"network_{suffix}.nc",
            )
            if os.path.exists(candidate):
                return candidate

    raise FileNotFoundError(f"Could not resolve network input: {path}")


scenario_network_paths = {
    name: resolve_network_input(path) for name, path in scenario_input_paths.items()
}
networks = {k: pypsa.Network(path) for k, path in scenario_network_paths.items()}

for name, path in scenario_network_paths.items():
    print(f"Loaded {name}: {path}")

In [ ]:
def robust_min_series(series_list):
    """Element-wise minimum across multiple Series, treating missing entries as zero."""
    return pd.concat(series_list, axis=1).fillna(0.0).min(axis=1)


def robust_min_dispatch(frames):
    """Element-wise minimum dispatch across scenarios for each snapshot and link."""
    merged = pd.concat(frames, axis=1, keys=range(len(frames))).fillna(0.0)
    return merged.T.groupby(level=1).min().T


trade_carrier = "shipping_hbi"
production_carrier = "hbi"

# Keep demand unchanged by starting from one solved scenario network.
robust_network = networks["chokepoints"].copy()

# Robust trade: minimum p_nom_opt for each HBI shipping connection.
trade_series = [
    net.links.loc[net.links.carrier == trade_carrier, "p_nom_opt"]
    for net in networks.values()
]
robust_trade = robust_min_series(trade_series)
robust_network.links.loc[robust_trade.index, "p_nom_opt"] = robust_trade.values

# Robust production: minimum realized HBI supply dispatch (not capacity).
hbi_link_names = robust_network.links.index[robust_network.links.carrier == production_carrier]

has_dispatch = all(
    hasattr(net, "links_t") and ("p1" in net.links_t) and (not net.links_t["p1"].empty)
    for net in networks.values()
)

if has_dispatch:
    # In these networks, delivered output at bus1 is represented by -p1 for Links.
    production_dispatch_frames = [
        (-net.links_t["p1"].loc[:, net.links.index[net.links.carrier == production_carrier]]).clip(lower=0)
        for net in networks.values()
    ]
    robust_dispatch = robust_min_dispatch(production_dispatch_frames)

    # Write back signed link flows expected by PyPSA: p1 negative for output delivery.
    robust_network.links_t["p1"].loc[:, robust_dispatch.columns] = -robust_dispatch.values

    if "p0" in robust_network.links_t:
        eff = robust_network.links.loc[robust_dispatch.columns, "efficiency"].replace(0, pd.NA)
        robust_network.links_t["p0"].loc[:, robust_dispatch.columns] = (
            robust_dispatch.div(eff, axis=1).fillna(0.0).values
        )

    # Keep a consistent capacity proxy from dispatch for diagnostics.
    robust_network.links.loc[robust_dispatch.columns, "p_nom_opt"] = robust_dispatch.max(axis=0).values
    robust_production = robust_dispatch.sum(axis=0)
else:
    # Fallback if dispatch time series are unavailable.
    production_series = [
        net.links.loc[net.links.carrier == production_carrier, "p_nom_opt"]
        for net in networks.values()
    ]
    robust_production = robust_min_series(production_series)
    robust_network.links.loc[robust_production.index, "p_nom_opt"] = robust_production.values

print(f"Robust trade links: {len(robust_trade)}")
print(f"Robust production links: {len(hbi_link_names)}")
print(f"Total robust HBI trade [Mt]: {robust_trade.sum() / 1e6:.2f}")
print(f"Total robust HBI production [Mt]: {robust_production.sum() / 1e6:.2f}")

In [ ]:
region_gdf = mt.build_region_geodataframe(snakemake.config)

mt.plot_trade_network(
    robust_network, #networks["constrain_supply"],
    product="hbi",
    alpha_supply=0.5,
    output_path=snakemake.output.robust_map_pdf,
    output_path_png=snakemake.output.robust_map_png,
    region_gdf=region_gdf,
)

print("Saved robust HBI map:")
print(f"  PDF: {snakemake.output.robust_map_pdf}")
print(f"  PNG: {snakemake.output.robust_map_png}")